# Phase 06A: Multi-Teacher Confirmation Entry & Protocol Lock

Trong đề tài này, “multi-teacher confirmation” được định nghĩa trước khi phân
tích là **xác nhận chéo trên ba họ Teacher và ba cặp matched Teacher–Student**:

- ExpansionNet v2 → S1 bằng Sequence KD;
- BLIP Base → S2 bằng Sequence KD;
- SigLIP ViT-B/16 → S3 bằng Logit KD.

Đây không phải phép cộng trực tiếp logits của ba Teacher: vocabulary, decoder và
knowledge channel không đồng nhất. Đơn vị bằng chứng là `cặp × seed`, tổng cộng
9 paired development runs. So sánh sử dụng **paired delta KD − baseline** trong
từng Student, không xếp hạng raw score giữa các Student khác kiến trúc.

Phase 06A chỉ khóa protocol và xác minh nguồn. Test vẫn `SEALED`.

## Inputs

1. Phase 05G complete/Phase 6 authorization.
2. Phase 04F ExpansionNet v2 → S1 three-seed aggregate.
3. Phase 04M BLIP → S2 three-seed aggregate.
4. Phase 04S SigLIP → S3 three-seed aggregate.

CPU, Internet Off, không dataset/model/GPU.


## Cell 1 — Authorization and source discovery


In [1]:
import csv, hashlib, json, math, os, time
from datetime import datetime, timezone
from pathlib import Path
STARTED=time.perf_counter()
IR=Path(os.environ.get("KLCN_INPUT_ROOT","/kaggle/input"))
NR=IR/"notebooks" if (IR/"notebooks").is_dir() else IR
OUT=Path(os.environ.get("KLCN_OUTPUT_ROOT","/kaggle/working//phase6_multi_teacher_confirmation_entry"))
AD=OUT/"audits";CD=OUT/"configs";RD=OUT/"results";MD=OUT/"manifests"
for d in [OUT,AD,CD,RD,MD]:d.mkdir(parents=True,exist_ok=True)
def now():return datetime.now(timezone.utc).isoformat()
def read(p):
    with Path(p).open("r",encoding="utf-8") as h:return json.load(h)
def write(p,x):
    p=Path(p);t=p.with_suffix(p.suffix+".tmp");t.write_text(json.dumps(x,ensure_ascii=False,indent=2)+"\n",encoding="utf-8");os.replace(t,p)
def sha(p):
    d=hashlib.sha256()
    with Path(p).open("rb") as h:
        for b in iter(lambda:h.read(8*1024*1024),b""):d.update(b)
    return d.hexdigest()
def one(n):
    m=sorted(NR.rglob(n));assert len(m)==1,(n,m);return m[0].resolve()
def inside(root,rel):
    root=Path(root).resolve();p=(root/rel).resolve();assert p.is_relative_to(root);return p
P5=one("PHASE5_SEQUENCE_KD_IMPROVEMENT_COMPLETE.json")
AUTH=one("PHASE6_MULTI_TEACHER_CONFIRMATION_ENTRY_AUTHORIZED.json")
A=one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json")
B=one("PHASE4_TEACHER_B_S2_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json")
C=one("PHASE4_TEACHER_C_S3_LOGIT_KD_THREE_SEED_AGGREGATE_COMPLETE.json")
p5,auth,a,b,c=map(read,[P5,AUTH,A,B,C])
assert p5["status"]=="complete" and p5["selected_arm"]=="R1" and p5["phase6_entry_authorized"] is True
assert auth["status"]=="authorized" and auth["phase5_selected_arm"]=="R1"
for x in [p5,auth,a,b,c]:
    assert x["test_set_status"]=="sealed" and x["test_set_used"] is False
for x in [a,b,c]:
    assert x["status"]=="complete" and x["reproduction_status"]=="passed"
    assert x["seeds"]==[42,123,2026] and x["cider_gate"]=="3 / 3"
write(AD/"phase6a_cell01_authorization.json",{"schema_version":"phase6a_cell01_v1","created_at_utc":now(),
 "status":"passed","teacher_families":3,"matched_pairs":3,"paired_runs":9,
 "dataset_read":False,"checkpoint_read":False,"training_started":False,
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 06A CELL 1 STATUS");print("="*72)
print("Status                    : PASSED")
print("Teacher families / pairs  : 3 / 3")
print("Paired development runs   : 9")
print("Dataset/checkpoint read   : False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 5 code cells")


PHASE 06A CELL 1 STATUS
Status                    : PASSED
Teacher families / pairs  : 3 / 3
Paired development runs   : 9
Dataset/checkpoint read   : False / False
Test Set status           : SEALED
Progress                  : 1 / 5 code cells


## Cell 2 — Verify aggregate manifests and summaries


In [2]:
STARTED=time.perf_counter()
def verify(marker_path,marker):
    ref=marker["manifest"]["csv"];p=inside(marker_path.parent,ref["relative_path"]);assert sha(p)==ref["sha256"]
    with p.open("r",encoding="utf-8",newline="") as h:rows=list(csv.DictReader(h))
    assert len(rows)==marker["manifest_rows"]
    for r in rows:
        f=inside(marker_path.parent,r["relative_path"]);assert f.is_file()
        assert f.stat().st_size==int(r["size_bytes"]) and sha(f)==r["sha256"]
    sref=marker["summary"];sp=inside(marker_path.parent,sref["relative_path"]);assert sha(sp)==sref["sha256"]
    return rows,read(sp)
arows,asum=verify(A,a);brows,bsum=verify(B,b);crows,csum=verify(C,c)
for s in [asum,bsum,csum]:
    assert s["seeds"]==[42,123,2026] and s["selection_split"]=="validation"
    assert s["test_set_status"]=="sealed" and s["test_set_used"] is False
write(AD/"phase6a_cell02_integrity.json",{"schema_version":"phase6a_cell02_v1","created_at_utc":now(),
 "status":"passed","manifest_rows_verified":len(arows)+len(brows)+len(crows),
 "summaries":"3 / 3 verified","seed_protocol":"exact [42,123,2026]",
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 06A CELL 2 STATUS");print("="*72)
print("Manifest rows verified    :",len(arows)+len(brows)+len(crows))
print("Aggregate summaries       : 3 / 3 VERIFIED")
print("Seed protocol             : [42,123,2026] EXACT")
print("Progress                  : 2 / 5 code cells")


PHASE 06A CELL 2 STATUS
Manifest rows verified    : 39
Aggregate summaries       : 3 / 3 VERIFIED
Seed protocol             : [42,123,2026] EXACT
Progress                  : 2 / 5 code cells


## Cell 3 — Build cross-teacher matched-pair evidence table


In [3]:
STARTED=time.perf_counter()
specs=[
 ("A→S1","ExpansionNet v2","S1 MobileNetV3-Large","Sequence KD alpha=0.5",asum),
 ("B→S2","BLIP Base","S2 MobileViT-S","Sequence KD",bsum),
 ("C→S3","SigLIP ViT-B/16","S3 TinyViT-5M","Logit KD",csum)]
rows=[];all_deltas=[]
for pair,t,s,m,z in specs:
    ag=z["aggregate"];per=z["per_seed"];d=[float(x["delta_cider_d"]) for x in per];all_deltas+=d
    rows.append({"pair":pair,"teacher":t,"student":s,"method":m,
      "baseline_cider_mean":ag["CIDEr-D"]["baseline"]["mean"],
      "kd_cider_mean":ag["CIDEr-D"]["kd"]["mean"],
      "delta_cider_mean":ag["CIDEr-D"]["paired_delta"]["mean"],
      "delta_cider_sample_std":ag["CIDEr-D"]["paired_delta"]["sample_std"],
      "baseline_bleu4_mean":ag["BLEU-4"]["baseline"]["mean"],
      "kd_bleu4_mean":ag["BLEU-4"]["kd"]["mean"],
      "delta_bleu4_mean":ag["BLEU-4"]["paired_delta"]["mean"],
      "baseline_ce_mean":ag["validation_ce"]["baseline"]["mean"],
      "kd_ce_mean":ag["validation_ce"]["kd"]["mean"],
      "delta_ce_mean":ag["validation_ce"]["paired_delta"]["mean"],
      "positive_cider_seeds":sum(x>0 for x in d),"seeds":len(d)})
assert len(all_deltas)==9 and all(x>0 for x in all_deltas)
csvp=RD/"phase6a_multi_teacher_evidence.csv"
with csvp.open("w",encoding="utf-8",newline="") as h:
    w=csv.DictWriter(h,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
write(RD/"phase6a_multi_teacher_evidence.json",{"schema_version":"phase6a_evidence_v1",
 "created_at_utc":now(),"rows":rows,"positive_cider_runs":9,"total_runs":9,
 "one_sided_sign_probability_descriptive":0.5**9,
 "warning":"Descriptive confirmation; pair/seed observations are not claimed independent.",
 "test_set_status":"sealed","test_set_used":False})
write(AD/"phase6a_cell03_evidence.json",{"schema_version":"phase6a_cell03_v1","created_at_utc":now(),
 "status":"passed","positive_cider_runs":"9 / 9","pair_mean_cider_gains":"3 / 3",
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 06A CELL 3 — MATCHED-PAIR EVIDENCE");print("="*72)
for r in rows:print(r["pair"],"| ΔCIDEr mean",round(r["delta_cider_mean"],6),"| wins",f'{r["positive_cider_seeds"]}/3')
print("Positive paired runs      : 9 / 9")
print("Progress                  : 3 / 5 code cells")


PHASE 06A CELL 3 — MATCHED-PAIR EVIDENCE
A→S1 | ΔCIDEr mean 0.120607 | wins 3/3
B→S2 | ΔCIDEr mean 0.027295 | wins 3/3
C→S3 | ΔCIDEr mean 0.020414 | wins 3/3
Positive paired runs      : 9 / 9
Progress                  : 3 / 5 code cells


## Cell 4 — Lock the multi-teacher confirmation protocol


In [4]:
STARTED=time.perf_counter()
protocol={"schema_version":"phase6a_multi_teacher_confirmation_protocol_v1","created_at_utc":now(),
 "status":"locked","interpretation":"cross-teacher matched-pair confirmation",
 "teachers":["ExpansionNet v2","BLIP Base","SigLIP ViT-B/16"],
 "students":["S1 MobileNetV3-Large","S2 MobileViT-S","S3 TinyViT-5M"],
 "knowledge_channels":["Sequence KD alpha=0.5","Sequence KD","Logit KD"],
 "unit_of_evidence":"matched Teacher-Student pair x seed","seeds":[42,123,2026],
 "total_paired_runs":9,"primary_endpoint":"paired delta CIDEr-D (KD - same-student baseline)",
 "confirmation_gate":{"positive_cider_runs":"9 / 9","positive_pair_mean_deltas":"3 / 3",
 "protocol_consistency_required":True},
 "raw_score_ranking_across_students":"prohibited",
 "raw_logit_fusion_across_teachers":"prohibited due incompatible token/decoder spaces",
 "new_training_required":False,"selection_split":"Validation only",
 "ce_role":"diagnostic; sign may differ by KD channel",
 "statistical_claim":"descriptive confirmation; n and dependence do not support a strong inferential claim",
 "test_set_status":"sealed","test_set_used":False}
PROTOCOL=CD/"phase6a_multi_teacher_confirmation_protocol.json";write(PROTOCOL,protocol)
write(AD/"phase6a_cell04_protocol.json",{"schema_version":"phase6a_cell04_v1","created_at_utc":now(),
 "status":"passed","protocol_locked":True,"raw_logit_fusion":False,"new_training_required":False,
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 06A CELL 4 — PROTOCOL LOCK");print("="*72)
print("Interpretation            : MATCHED-PAIR CROSS-TEACHER CONFIRMATION")
print("Raw logit fusion          : PROHIBITED")
print("Primary evidence          : paired ΔCIDEr-D")
print("New training required     : False")
print("Progress                  : 4 / 5 code cells")


PHASE 06A CELL 4 — PROTOCOL LOCK
Interpretation            : MATCHED-PAIR CROSS-TEACHER CONFIRMATION
Raw logit fusion          : PROHIBITED
Primary evidence          : paired ΔCIDEr-D
New training required     : False
Progress                  : 4 / 5 code cells


## Cell 5 — Package and authorize Phase 06B evidence synthesis


In [5]:
STARTED=time.perf_counter()
files=sorted(AD.glob("*.json"))+sorted(CD.glob("*.json"))+sorted(RD.glob("*"));mrs=[]
for p in files:mrs.append({"relative_path":p.relative_to(OUT).as_posix(),"size_bytes":p.stat().st_size,"sha256":sha(p)})
mp=MD/"phase6a_manifest.csv"
with mp.open("w",encoding="utf-8",newline="") as h:
    w=csv.DictWriter(h,fieldnames=["relative_path","size_bytes","sha256"]);w.writeheader();w.writerows(mrs)
sp=MD/"SHA256SUMS";sp.write_text("".join(f"{r['sha256']}  {r['relative_path']}\n" for r in mrs),encoding="utf-8")
authp=OUT/"PHASE6B_MULTI_TEACHER_EVIDENCE_SYNTHESIS_AUTHORIZED.json"
write(authp,{"schema_version":"phase6b_evidence_synthesis_authorized_v1","created_at_utc":now(),
 "status":"authorized","protocol":{"relative_path":PROTOCOL.relative_to(OUT).as_posix(),"sha256":sha(PROTOCOL)},
 "required_next":"Phase 06B cross-teacher statistical/descriptive synthesis and final development conclusion",
 "new_training_authorized":False,"test_set_status":"sealed","test_set_used":False})
marker=OUT/"PHASE6A_MULTI_TEACHER_CONFIRMATION_ENTRY_COMPLETE.json"
write(marker,{"schema_version":"phase6a_multi_teacher_entry_complete_v1","created_at_utc":now(),
 "status":"complete","teacher_families":3,"matched_pairs":3,"paired_runs":9,
 "positive_cider_runs":"9 / 9","positive_pair_mean_deltas":"3 / 3",
 "protocol_locked":True,"phase6b_authorized":True,"phase6b_authorization_sha256":sha(authp),
 "manifest":{"csv":{"relative_path":mp.relative_to(OUT).as_posix(),"sha256":sha(mp),"rows":len(mrs)},
 "sha256sums":{"relative_path":sp.relative_to(OUT).as_posix(),"sha256":sha(sp)}},
 "training_started":False,"test_set_status":"sealed","test_set_used":False})
print("="*72);print("PHASE 06A FINAL STATUS");print("="*72)
print("Status                    : COMPLETE")
print("Teacher families          : 3 / 3")
print("Positive CIDEr runs       : 9 / 9")
print("Protocol                  : LOCKED")
print("New training              : NOT REQUIRED")
print("Phase 06B                 : AUTHORIZED")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 5 code cells")


PHASE 06A FINAL STATUS
Status                    : COMPLETE
Teacher families          : 3 / 3
Positive CIDEr runs       : 9 / 9
Protocol                  : LOCKED
New training              : NOT REQUIRED
Phase 06B                 : AUTHORIZED
Test Set status           : SEALED
Progress                  : 5 / 5 code cells
